# CarbonGlobe — FusionED lite · 2×T4 distributed · resumable across sessions

**IEEE Big Data Cup 2026 — CarbonGlobe AI Emulation Challenge**

Same FusionED idea as `carbonglobe-fusion-v2.ipynb` (LSTM-XL ⊕ EcoFormer, per-variable gated blend of their
residual deltas) and the same recipe (scheduled-sampling rollout, EMA, clamp, early stopping + LR schedule on
the real 40-year metric), rebuilt so it actually runs on Kaggle:

| | fusion-v2 | this notebook |
|---|---|---|
| parameters | 43.4M (`PRESET='full'`) | **10.9M** (`PRESET='lite'`, default) |
| GPUs | 1 | **both T4s** (`tf.distribute.MirroredStrategy`, gradients all-reduced) |
| rollout training graph | K steps unrolled in Python (up to 16 model copies, 5 traces) | **`tf.while_loop`**, model traced once |
| 12-hour limit | training lost when the session ends | **checkpoint every epoch, auto-resume next session** |
| seeds | multi-seed loop | 1 seed (run again with another `SEED`/`RUN_NAME` and blend) |

Each session also writes `submission.csv` from the best weights so far, so every run is submittable.

## How to run it on Kaggle (multi-session)

1. **Settings**: Accelerator = **GPU T4 x2**, Internet = on (for `git clone`). Inputs: the competition data and
   the `zhihaow/carbonglobe` dataset. If the repo is private, add the Kaggle Secret `GITHUB_TOKEN`.
2. **Session 1**: *Save Version → Save & Run All (Commit)*. Training stops by itself about
   `SESSION_LIMIT_H − FINAL_RESERVE_MIN` hours after start, then writes `submission.csv`,
   `fusion_lite_preds.npz`, and the checkpoint folder **`fusion_lite_ckpt/`**.
3. **Next sessions**: make that output available as an input — e.g. on the finished version's *Output* tab
   create a dataset from it (later sessions: add a new dataset version), and attach it. Run again: the notebook
   finds `fusion_lite_ckpt/state.json` anywhere under `/kaggle/input` (newest epoch wins) and continues at the
   next epoch with the same weights, EMA, optimizer moments, LR schedule and early-stopping counters.
   `RESUME_FROM` lets you point at a folder explicitly.
4. It stops on early stopping or `MAX_EPOCHS` (both counted across all sessions); further runs then only rebuild
   the submission from the best weights. **To train longer**, raise `MAX_EPOCHS` (or `ES_PATIENCE`) and run
   again with the checkpoint attached: the run re-opens and continues from where it stopped.

Keep `PRESET` and `BACKBONE` the same across sessions (the checkpoint refuses to load otherwise). LR, curriculum,
time budget and batch size may change between sessions. Kaggle's weekly GPU quota still applies.

## 0. Clone repo & imports

In [ ]:
import time
NB_T0 = time.time()   # session clock: the time budget is measured from here

import os, sys, subprocess, time, gc

REPO_DIR = '/kaggle/working/carbon-globe'
REPO_URL = 'github.com/kuongan/carbon-globe.git'

def _github_token():
    # Store the token in a Kaggle Secret named GITHUB_TOKEN (Add-ons -> Secrets).
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        return os.environ.get('GITHUB_TOKEN', '')

if not os.path.isdir(REPO_DIR):
    tok = _github_token()
    url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr.replace(tok, '***') if tok else r.stderr
        raise RuntimeError('git clone failed:\n' + err)

PKG_DIR = os.path.join(REPO_DIR, 'LSTM-LSTNet-DeepED')
sys.path.insert(0, PKG_DIR)
os.chdir(PKG_DIR)
print('cwd =', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm

from configs.constant_glob import N_CONS, N_OUT, N_YEAR, N_T, N_FEA, TREE_BAND
from utils.func import normData, invNormData
from models.model import MM_LSTM_Age_v2   # param-count reference

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| Keras', keras.__version__)
print('N_CONS=%d N_OUT=%d N_FEA=%d N_YEAR=%d N_T=%d' % (N_CONS, N_OUT, N_FEA, N_YEAR, N_T))
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print('memory growth note:', e)
import json, glob, shutil

## 1. Constants, stats & competition metric

In [ ]:
COMP = '/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public'
BASE = '/kaggle/input/datasets/zhihaow/carbonglobe'
OUT  = '/kaggle/working'

SEED_AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
SCENARIOS = ['ssp126', 'ssp585']
N_AGES, N_AGE_FEA = len(SEED_AGES), 15

stats = np.load(f'{BASE}/data_stats/data_stats.npz')
X_MEAN, X_STD = stats['x_mean'].astype(np.float32), stats['x_std'].astype(np.float32)
Y_MEAN, Y_STD = stats['y_mean'].astype(np.float32), stats['y_std'].astype(np.float32)

agetri = np.load(f'{BASE}/data_stats/age_triplet.npz')
TRI_AGE   = agetri['age'].astype(np.float32)
TRI_MEAN  = agetri['age_mean'].astype(np.float32)
TRI_SLOPE = agetri['age_slope'].astype(np.float32)
assert list(TRI_AGE.astype(int)) == SEED_AGES

AGE_MEAN_N  = normData(TRI_MEAN,  Y_MEAN, Y_STD)
AGE_SLOPE_N = normData(TRI_SLOPE, Y_MEAN, Y_STD)
AGE_FEAT = np.stack([np.concatenate([[TRI_AGE[i] / 500.0], AGE_MEAN_N[i], AGE_SLOPE_N[i]])
                     for i in range(N_AGES)]).astype(np.float32)
FLOOR_N = normData(np.zeros(N_OUT, np.float32), Y_MEAN, Y_STD).astype(np.float32)
print('Y_MEAN =', np.round(Y_MEAN, 5))
print('submission scale -> height /%.5f , agb /%.5f' % (Y_MEAN[0], Y_MEAN[1]))


def scored_metric(pred, true):
    """The competition metric on year-40 height & AGB (physical units in, scaled MSE out)."""
    ph, th = pred[..., 0] / Y_MEAN[0], true[..., 0] / Y_MEAN[0]
    pa, ta = pred[..., 1] / Y_MEAN[1], true[..., 1] / Y_MEAN[1]
    mse_h = float(np.mean((ph - th) ** 2)); mse_a = float(np.mean((pa - ta) ** 2))
    return 0.5 * (mse_h + mse_a), mse_h, mse_a

## 2. Load train / val arrays (standardized in place)

In [ ]:
t0 = time.time()
res = np.load(f'{BASE}/data_global/res_train4_test8.npz', mmap_mode='r')
x_train = np.asarray(res['x_train'], dtype=np.float32)
x_val   = np.asarray(res['x_test'],  dtype=np.float32)
y_train = np.ascontiguousarray(res['y_train'][..., 11, :]).astype(np.float32)
y_val   = np.ascontiguousarray(res['y_test'][...,  11, :]).astype(np.float32)
res.close(); gc.collect()

def standardize_inplace(x):
    if not x.flags.writeable: x = x.copy()
    x -= X_MEAN; x /= (X_STD + 1e-10)
    return x

x_train   = standardize_inplace(x_train)
x_val     = standardize_inplace(x_val)
y_train_n = normData(y_train, Y_MEAN, Y_STD).astype(np.float32)
del y_train; gc.collect()

val_y0   = y_val[:, :, 0, :]
val_true = y_val[:, :, 40, :]
N_TRAIN  = x_train.shape[0]
N_PAIRS  = N_AGES * N_YEAR * N_TRAIN
print('loaded in %.1fs | x_train' % (time.time() - t0), x_train.shape, '| x_val', x_val.shape)

## 3. Models — LSTM-XL, EcoFormer, FusionED (identical code to fusion-v2)

In [ ]:
# ---------------- LSTM-XL ----------------
class GatedBranch(layers.Layer):
    def __init__(self, idx, units, mlp_units, dropout, **kw):
        super().__init__(name=f'branch{idx}', **kw)
        self.idx = idx
        self.lstm_data = layers.LSTM(units, return_sequences=True)
        self.lstm_gate = layers.LSTM(units, return_sequences=True)
        self.gate      = layers.Dense(units, activation='sigmoid')
        self.lstm_out  = layers.LSTM(units, return_sequences=True)
        self.norm      = layers.LayerNormalization(epsilon=1e-5)
        self.pool      = layers.Dense(1)
        self.mlp1      = layers.Dense(mlp_units, activation='gelu')
        self.mlp2      = layers.Dense(128, activation='gelu')
        self.drop      = layers.Dropout(dropout)
        self.head      = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros')
    def call(self, h, prev, age, training=False):
        i = self.idx; T = tf.shape(h)[1]
        age_i = tf.concat([age[:, :1], age[:, i+1:i+2], age[:, i+8:i+9]], axis=-1)
        loc = tf.concat([prev[:, i:i+1], age_i], axis=-1)
        z = tf.concat([h, tf.repeat(loc[:, None, :], T, axis=1)], axis=-1)
        s = self.lstm_data(z) * self.gate(self.lstm_gate(z))
        s = self.norm(self.lstm_out(s))
        w = tf.nn.softmax(self.pool(s), axis=1)
        feat = tf.concat([tf.reduce_sum(w * s, axis=1), s[:, -1]], axis=-1)
        feat = self.mlp2(self.drop(self.mlp1(feat), training=training))
        feat = tf.concat([feat, prev, age_i], axis=-1)
        return prev[:, i:i+1] + self.head(feat)

class LSTMXLBackbone(keras.Model):
    """Returns the DELTA (B,7) so it can be fused; add prev outside for standalone use."""
    def __init__(self, n_out=N_OUT, n_cons=N_CONS, n_age_fea=N_AGE_FEA, d_model=320, n_shared=3,
                 branch_units=320, mlp_units=512, dropout=0.1, return_delta=False, **kw):
        super().__init__(**kw)
        self.n_out, self.n_cons, self.n_age_fea, self.return_delta = n_out, n_cons, n_age_fea, return_delta
        self.ctx_proj = layers.Dense(128, activation='gelu')
        self.in_proj  = layers.Dense(d_model)
        self.in_norm  = layers.LayerNormalization(epsilon=1e-5)
        self.enc_lstms = [layers.Bidirectional(layers.LSTM(d_model // 2, return_sequences=True)) for _ in range(n_shared)]
        self.enc_norms = [layers.LayerNormalization(epsilon=1e-5) for _ in range(n_shared)]
        self.enc_drop  = layers.Dropout(dropout)
        self.branches  = [GatedBranch(i, branch_units, mlp_units, dropout) for i in range(n_out)]
    def call(self, inputs, training=False):
        nc, no, na = self.n_cons, self.n_out, self.n_age_fea
        env = inputs[:, :, :nc]; prev = inputs[:, -1, nc:nc+no]; age = inputs[:, -1, nc+no:nc+no+na]
        T = tf.shape(env)[1]
        ctx = self.ctx_proj(tf.concat([prev, age], axis=-1))
        h = tf.concat([env, tf.repeat(ctx[:, None, :], T, axis=1)], axis=-1)
        h = self.in_norm(self.in_proj(h))
        for lstm, norm in zip(self.enc_lstms, self.enc_norms):
            h = norm(h + self.enc_drop(lstm(h), training=training))
        out = tf.concat([b(h, prev, age, training=training) for b in self.branches], axis=-1)  # prev+delta
        return (out - prev) if self.return_delta else out

# ---------------- EcoFormer ----------------
class EcoEncBlock(layers.Layer):
    def __init__(self, d, heads, d_ff, drop, **kw):
        super().__init__(**kw)
        self.n1 = layers.LayerNormalization(epsilon=1e-5)
        self.att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.n2 = layers.LayerNormalization(epsilon=1e-5)
        self.f1 = layers.Dense(d_ff, activation='gelu'); self.f2 = layers.Dense(d)
        self.drop = layers.Dropout(drop)
    def call(self, x, training=False):
        y = self.n1(x); x = x + self.drop(self.att(y, y, training=training), training=training)
        y = self.n2(x); return x + self.drop(self.f2(self.f1(y)), training=training)

class EcoDecBlock(layers.Layer):
    def __init__(self, d, heads, d_ff, drop, **kw):
        super().__init__(**kw)
        self.n1 = layers.LayerNormalization(epsilon=1e-5)
        self.self_att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.n2 = layers.LayerNormalization(epsilon=1e-5)
        self.cross_att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.n3 = layers.LayerNormalization(epsilon=1e-5)
        self.f1 = layers.Dense(d_ff, activation='gelu'); self.f2 = layers.Dense(d)
        self.drop = layers.Dropout(drop)
    def call(self, q, mem, training=False):
        y = self.n1(q); q = q + self.drop(self.self_att(y, y, training=training), training=training)
        y = self.n2(q); q = q + self.drop(self.cross_att(y, mem, training=training), training=training)
        y = self.n3(q); return q + self.drop(self.f2(self.f1(y)), training=training)

class EcoFormerBackbone(keras.Model):
    def __init__(self, n_out=N_OUT, n_cons=N_CONS, n_age_fea=N_AGE_FEA, n_months=N_T, d_model=384,
                 n_heads=8, d_ff=1536, n_enc=6, n_dec=4, dropout=0.1, return_delta=False, **kw):
        super().__init__(**kw)
        self.n_out, self.n_cons, self.n_age_fea, self.n_months = n_out, n_cons, n_age_fea, n_months
        self.return_delta = return_delta
        self.env_proj  = layers.Dense(d_model); self.env_norm = layers.LayerNormalization(epsilon=1e-5)
        self.month_emb = layers.Embedding(n_months, d_model)
        self.state_in1 = layers.Dense(d_model, activation='gelu'); self.state_in2 = layers.Dense(d_model)
        self.enc = [EcoEncBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc)]
        self.enc_norm = layers.LayerNormalization(epsilon=1e-5)
        self.var_emb = layers.Embedding(n_out, d_model)
        self.var_in1 = layers.Dense(d_model, activation='gelu'); self.var_in2 = layers.Dense(d_model)
        self.dec = [EcoDecBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_dec)]
        self.dec_norm = layers.LayerNormalization(epsilon=1e-5)
        self.head1 = layers.Dense(d_model, activation='gelu')
        self.head2 = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros')
    def call(self, inputs, training=False):
        nc, no, na = self.n_cons, self.n_out, self.n_age_fea
        env = inputs[:, :, :nc]; prev = inputs[:, -1, nc:nc+no]; age = inputs[:, -1, nc+no:nc+no+na]
        h = self.env_norm(self.env_proj(env)) + self.month_emb(tf.range(self.n_months))[None]
        s = self.state_in2(self.state_in1(tf.concat([prev, age], axis=-1)))[:, None, :]
        h = tf.concat([s, h], axis=1)
        for blk in self.enc: h = blk(h, training=training)
        mem = self.enc_norm(h)
        age_tri = tf.stack([tf.stack([age[:, 0], age[:, 1+i], age[:, 8+i]], axis=-1) for i in range(no)], axis=1)
        vf = tf.concat([prev[..., None], age_tri], axis=-1)
        q = self.var_in2(self.var_in1(vf)) + self.var_emb(tf.range(no))[None]
        for blk in self.dec: q = blk(q, mem, training=training)
        delta = self.head2(self.head1(self.dec_norm(q)))[..., 0]
        return delta if self.return_delta else (prev + delta)

# ---------------- Fusion ----------------
class FusionED(keras.Model):
    """Run both backbones, blend their per-variable deltas with a learned gate, residual-add."""
    def __init__(self, lstm_kw=None, eco_kw=None, **kw):
        super().__init__(**kw)
        self.lstm = LSTMXLBackbone(return_delta=True, **(lstm_kw or {}))
        self.eco  = EcoFormerBackbone(return_delta=True, **(eco_kw or {}))
        # per-variable gate in [0,1]; init 0 -> sigmoid 0.5 (equal blend)
        self.gate = tf.Variable(tf.zeros((N_OUT,)), trainable=True, name='fuse_gate')
    def call(self, inputs, training=False):
        prev = inputs[:, -1, N_CONS:N_CONS+N_OUT]
        d_lstm = self.lstm(inputs, training=training)
        d_eco  = self.eco(inputs, training=training)
        g = tf.sigmoid(self.gate)[None, :]
        return prev + g * d_lstm + (1.0 - g) * d_eco

print('models defined: LSTMXLBackbone, EcoFormerBackbone, FusionED')

## 4. Config

In [ ]:
RUN_NAME = 'fusion_lite'

# ---- model ----
PRESET   = 'lite'          # 'lite' (default) | 'full' (= fusion-v2 sizes; ~4x the compute)
BACKBONE = 'fusion'        # 'fusion' | 'lstm_xl' | 'ecoformer'
PRESETS = {
    'lite': dict(lstm_kw=dict(d_model=192, n_shared=2, branch_units=160, mlp_units=256, dropout=0.1),
                 eco_kw=dict(d_model=256, n_heads=8, d_ff=768, n_enc=4, n_dec=2, dropout=0.1)),
    'full': dict(lstm_kw=dict(d_model=320, n_shared=3, branch_units=320, mlp_units=512, dropout=0.1),
                 eco_kw=dict(d_model=384, n_heads=8, d_ff=1536, n_enc=6, n_dec=4, dropout=0.1)),
}

# ---- hardware ----
USE_MULTI_GPU = True       # MirroredStrategy over all visible GPUs (falls back to 1 device automatically)
CROSS_DEVICE  = 'nccl'     # 'nccl' | 'hierarchical' (switch if the first step hangs or NCCL errors)
UNROLL_LOOP   = False      # False: K-year rollout as tf.while_loop (compiled once)
                           # True : python-unrolled loop of the original engine (slower compile, more memory)

# ---- optimisation (batch sizes are GLOBAL = summed over GPUs) ----
BATCH_SIZE   = 1024        # global batch for K=1
ROLL_TOKENS  = 4096        # cap on global batch * K (memory of back-prop through the rollout)
MIN_BATCH    = 64
BASE_LR      = 4e-4
MIN_LR       = 1e-5
WARMUP_STEPS = 1500
CLIP_NORM    = 1.0         # applied to the all-reduced (global) gradient
WEIGHT_DECAY = 1e-2
Y_W          = [2., 2., 1., 1., 1., 1., 1.]
CLAMP_NONNEG = True

# ---- epochs (counted across ALL sessions) / early stop / LR plateau ----
MAX_EPOCHS        = 30
SAMPLES_PER_EPOCH = 800_000      # model-steps per epoch (windows = this // K)
LR_FACTOR, LR_PATIENCE = 0.5, 2
ES_PATIENCE       = 5
MIN_DELTA         = 1e-4

# ---- scheduled-sampling curriculum: (start_epoch, K years/window, teacher-forcing prob) ----
CURRICULUM = [(1, 1, 1.00), (3, 4, 0.75), (6, 8, 0.50), (9, 12, 0.25), (12, 16, 0.0)]

USE_EMA   = True
EMA_DECAY = 0.999

# ---- sessions / resume ----
SESSION_LIMIT_H   = 11.0   # Kaggle stops a session at 12 h; stay below it
FINAL_RESERVE_MIN = 30     # kept free for the final val + test rollout and writing outputs
RESUME            = True   # continue from the newest {RUN_NAME}_ckpt under /kaggle/input or /kaggle/working
RESUME_FROM       = ''     # optional explicit path to a {RUN_NAME}_ckpt folder

def make_model():
    p = PRESETS[PRESET]
    if BACKBONE == 'lstm_xl':
        return LSTMXLBackbone(**p['lstm_kw'])
    if BACKBONE == 'ecoformer':
        return EcoFormerBackbone(**p['eco_kw'])
    return FusionED(lstm_kw=p['lstm_kw'], eco_kw=p['eco_kw'])

def make_optimizer(lr):
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY)
    try:
        opt.exclude_from_weight_decay(var_names=['bias', 'gamma', 'beta', 'layer_normalization',
                                                 'embeddings', 'fuse_gate'])
    except Exception as e:
        print('exclude_from_weight_decay note:', e)
    return opt

print(f'{RUN_NAME}: PRESET={PRESET} BACKBONE={BACKBONE} | curriculum {CURRICULUM} | '
      f'session limit {SESSION_LIMIT_H} h')

## 5. Distribution strategy & model build

With two GPUs every training step splits the global batch in half, runs both halves in parallel, and all-reduces
the gradients. Clipping is applied to the summed gradient, so a 2-GPU step does the same update as one GPU on the
full batch.

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if USE_MULTI_GPU and len(gpus) > 1:
    _cdo = (tf.distribute.NcclAllReduce() if CROSS_DEVICE == 'nccl'
            else tf.distribute.HierarchicalCopyAllReduce())
    STRATEGY = tf.distribute.MirroredStrategy(cross_device_ops=_cdo)
else:
    STRATEGY = tf.distribute.get_strategy()      # single GPU / CPU
NREP = STRATEGY.num_replicas_in_sync
print('GPUs visible:', len(gpus), '| replicas in sync:', NREP, '|', type(STRATEGY).__name__)

In [ ]:
with STRATEGY.scope():
    MODEL = make_model()
    _ = MODEL(tf.zeros((2, N_T, N_FEA), tf.float32), training=False)     # build all variables
    OPT = make_optimizer(BASE_LR)
    OPT.build(MODEL.trainable_variables)
    TVARS = list(MODEL.trainable_variables)
    EMA = ([tf.Variable(v.numpy(), trainable=False, name=f'ema_{i}') for i, v in enumerate(TVARS)]
           if USE_EMA else [])

n_params = MODEL.count_params()
_ref = MM_LSTM_Age_v2(n_out=N_OUT, n_cons=N_CONS)
_ = _ref(tf.zeros((2, N_T, N_FEA), tf.float32), training=False)
print(f'{RUN_NAME}/{BACKBONE}/{PRESET}: {n_params/1e6:.2f}M params '
      f'(MM_LSTM_Age_v2 reference {_ref.count_params()/1e6:.2f}M)')
del _ref; gc.collect()

## 6. Training engine

* window sampler (vectorized numpy gather) → `tf.data` → split across replicas;
* K-year rollout loss with scheduled sampling, gradients through the fed-back predictions;
* distributed step: per-replica gradients → all-reduce → global-norm clip → AdamW, then EMA update;
* 40-year validation/test rollout, LR warm-up + ReduceLROnPlateau, all with savable state.

In [ ]:
FLOOR_N = normData(np.zeros(N_OUT, np.float32), Y_MEAN, Y_STD).astype(np.float32)
FLOOR_T = tf.constant(FLOOR_N)
Y_W_T   = tf.constant(Y_W, tf.float32)

# ---------------------------------------------------------------- data
def global_batch_for(K):
    b = int(min(BATCH_SIZE, max(MIN_BATCH, ROLL_TOKENS // K)))
    return max(NREP, b - b % NREP)                   # divisible by the number of replicas

def window_batches(K, n_windows, batch, rng):
    """Yield standardized (env (B,K,12,136), states (B,K+1,7), age_feat (B,15))."""
    T0 = N_YEAR - K + 1
    total = N_AGES * N_TRAIN * T0
    n = min(n_windows, total)
    idx = rng.choice(total, size=n, replace=False)
    kx, ky = np.arange(K), np.arange(K + 1)
    for s in range(0, n - batch + 1, batch):
        b = idx[s:s + batch]
        ai, rem = np.divmod(b, N_TRAIN * T0)
        site, t0 = np.divmod(rem, T0)
        env = x_train[site[:, None], t0[:, None] + kx]
        yy  = y_train_n[ai[:, None], site[:, None], t0[:, None] + ky]
        yield env, yy, AGE_FEAT[ai]

def make_dataset(K, n_windows, batch, rng):
    sig = (tf.TensorSpec((batch, K, N_T, N_CONS), tf.float32),
           tf.TensorSpec((batch, K + 1, N_OUT), tf.float32),
           tf.TensorSpec((batch, N_AGE_FEA), tf.float32))
    ds = tf.data.Dataset.from_generator(lambda: window_batches(K, n_windows, batch, rng),
                                        output_signature=sig).prefetch(8)
    opts = tf.data.Options()
    opts.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.OFF
    return STRATEGY.experimental_distribute_dataset(ds.with_options(opts))

# ---------------------------------------------------------------- rollout loss (per replica)
def _predict(env_k, prev, af_m):
    prev_m = tf.repeat(prev[:, None, :], N_T, axis=1)
    pred = MODEL(tf.concat([env_k, prev_m, af_m], axis=-1), training=True)
    return tf.maximum(pred, FLOOR_T) if CLAMP_NONNEG else pred

def rollout_loss_unrolled(env, yy, af, tf_ratio):
    """Original engine: python loop over a static K."""
    K = env.shape[1]; b = tf.shape(env)[0]
    af_m = tf.repeat(af[:, None, :], N_T, axis=1)
    prev, loss = yy[:, 0], 0.0
    for k in range(K):
        pred = _predict(env[:, k], prev, af_m)
        loss += tf.reduce_mean(Y_W_T * tf.square(pred - yy[:, k + 1]))
        if k < K - 1:
            keep = tf.cast(tf.random.uniform((b, 1)) < tf_ratio, tf.float32)
            prev = keep * yy[:, k + 1] + (1.0 - keep) * pred
    return loss / float(K)

def rollout_loss_while(env, yy, af, tf_ratio):
    """Same maths as rollout_loss_unrolled, but the model is traced once for any K."""
    K = tf.shape(env)[1]; b = tf.shape(env)[0]
    af_m = tf.repeat(af[:, None, :], N_T, axis=1)
    def body(k, prev, loss):
        tgt  = tf.gather(yy, k + 1, axis=1)
        pred = _predict(tf.gather(env, k, axis=1), prev, af_m)
        loss = loss + tf.reduce_mean(Y_W_T * tf.square(pred - tgt))
        keep = tf.cast(tf.random.uniform((b, 1)) < tf_ratio, tf.float32)
        return k + 1, keep * tgt + (1.0 - keep) * pred, loss
    _, _, loss = tf.while_loop(lambda k, p, l: k < K, body,
                               (tf.constant(0), yy[:, 0], tf.constant(0.0)))
    return loss / tf.cast(K, tf.float32)

ROLLOUT_LOSS = rollout_loss_unrolled if UNROLL_LOOP else rollout_loss_while

# ---------------------------------------------------------------- distributed step
def replica_step(env, yy, af, tf_ratio):
    with tf.GradientTape() as tape:
        loss = ROLLOUT_LOSS(env, yy, af, tf_ratio) / NREP      # local mean / replicas
    grads = tape.gradient(loss, TVARS)
    grads = [tf.zeros_like(v) if g is None else tf.convert_to_tensor(g) for g, v in zip(grads, TVARS)]
    if NREP > 1:
        grads = tf.distribute.get_replica_context().all_reduce(tf.distribute.ReduceOp.SUM, grads)
        grads, gnorm = tf.clip_by_global_norm(grads, CLIP_NORM)   # clip the GLOBAL gradient
        grads = [g / NREP for g in grads]                          # the optimizer sums replicas again
    else:
        grads, gnorm = tf.clip_by_global_norm(grads, CLIP_NORM)
    OPT.apply_gradients(zip(grads, TVARS))
    return loss, gnorm

def _dist_step(env, yy, af, tf_ratio):
    loss, gnorm = STRATEGY.run(replica_step, args=(env, yy, af, tf_ratio))
    loss  = STRATEGY.reduce(tf.distribute.ReduceOp.SUM, loss, axis=None)
    gnorm = STRATEGY.reduce(tf.distribute.ReduceOp.MEAN, gnorm, axis=None)
    for e, v in zip(EMA, TVARS):
        e.assign(EMA_DECAY * e + (1.0 - EMA_DECAY) * v)
    return loss, gnorm

dist_step = tf.function(_dist_step, reduce_retracing=not UNROLL_LOOP)

# ---------------------------------------------------------------- inference / metric
@tf.function(reduce_retracing=True)
def infer(x):
    return MODEL(x, training=False)

def rollout(x, y0_raw, x_is_norm=False, desc=None, bs=8192):
    """40-year autoregressive rollout, all seed ages at once. Returns year-40 state (15,N,7), physical."""
    N, A = x.shape[0], y0_raw.shape[0]
    prev = normData(y0_raw, Y_MEAN, Y_STD).reshape(A * N, N_OUT).astype(np.float32)
    site_idx = np.tile(np.arange(N), A)
    af_rows = AGE_FEAT[np.repeat(np.arange(A), N)]
    years = tqdm(range(N_YEAR), desc=desc, leave=False) if desc else range(N_YEAR)
    for t in years:
        env_t = x[:, t] if x_is_norm else normData(x[:, t], X_MEAN, X_STD).astype(np.float32)
        nxt = np.empty_like(prev)
        for s in range(0, A * N, bs):
            e = min(s + bs, A * N)
            xb = np.concatenate([env_t[site_idx[s:e]],
                                 np.repeat(prev[s:e, None, :], N_T, axis=1),
                                 np.repeat(af_rows[s:e, None, :], N_T, axis=1)], axis=-1)
            p = infer(tf.constant(xb, tf.float32)).numpy()
            nxt[s:e] = np.maximum(p, FLOOR_N) if CLAMP_NONNEG else p
        prev = nxt
    out = invNormData(prev, Y_MEAN, Y_STD).reshape(A, N, N_OUT)
    return np.maximum(out, 0.0) if CLAMP_NONNEG else out

def get_tvars():
    return [v.numpy() for v in TVARS]

def set_tvars(ws):
    for v, w in zip(TVARS, ws):
        v.assign(w)

# ---------------------------------------------------------------- LR schedule (state is checkpointed)
def set_lr(v):
    OPT.learning_rate = float(v)

class LRController:
    """Linear warm-up per step, then ReduceLROnPlateau per epoch on the validation metric."""
    def __init__(self):
        self.step, self.best, self.wait = 0, np.inf, 0
        self.lr = BASE_LR * (0.1 if WARMUP_STEPS > 0 else 1.0)
        set_lr(self.lr)
    def on_step(self):
        if self.step < WARMUP_STEPS:
            self.lr = BASE_LR * (0.1 + 0.9 * (self.step + 1) / WARMUP_STEPS)
            set_lr(self.lr)
        self.step += 1
    def on_epoch(self, metric, counting=True):
        if metric < self.best - MIN_DELTA:
            self.best, self.wait = metric, 0
        elif counting and self.step >= WARMUP_STEPS:
            self.wait += 1
            if self.wait >= LR_PATIENCE and self.lr > MIN_LR:
                self.lr = max(self.lr * LR_FACTOR, MIN_LR); set_lr(self.lr); self.wait = 0
                print(f'    ReduceLROnPlateau -> lr = {self.lr:.2e}')
    def state(self):
        return dict(step=self.step, best=None if not np.isfinite(self.best) else float(self.best),
                    wait=self.wait, lr=float(self.lr))
    def load(self, d):
        self.step, self.wait, self.lr = int(d['step']), int(d['wait']), float(d['lr'])
        self.best = np.inf if d['best'] is None else float(d['best'])
        set_lr(self.lr)

def stage_for_epoch(ep):
    si = 0
    for j, (start, _, _) in enumerate(CURRICULUM):
        if ep >= start:
            si = j
    _, K, tfr = CURRICULUM[si]
    return si, K, tfr

print('engine ready | rollout loss:', ROLLOUT_LOSS.__name__,
      '| global batch per K:', {K: global_batch_for(K) for _, K, _ in CURRICULUM})

## 7. Checkpoint & resume

`<RUN_NAME>_ckpt/` holds `last.npz` (weights, EMA, optimizer moments), `best.npz` (best weights by the 40-year
validation metric) and `state.json` (epoch counter, LR schedule, early-stopping counters, history). Files are
written to a temp name and renamed, so a cut-off write never corrupts the previous checkpoint.

In [ ]:
CKPT_DIR = os.path.join(OUT, f'{RUN_NAME}_ckpt')
FINGERPRINT = dict(run=RUN_NAME, preset=PRESET, backbone=BACKBONE, n_params=int(MODEL.count_params()),
                   n_tvars=len(TVARS), n_opt_vars=len(OPT.variables))

def _atomic_savez(path, arrs):
    tmp = path[:-4] + '.tmp.npz'
    np.savez(tmp, **arrs)
    os.replace(tmp, path)

def _atomic_json(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)

def save_ckpt(state, best_weights=None):
    os.makedirs(CKPT_DIR, exist_ok=True)
    arrs = {f'm{i}': w for i, w in enumerate(MODEL.get_weights())}
    arrs.update({f'e{i}': e.numpy() for i, e in enumerate(EMA)})
    arrs.update({f'o{i}': v.numpy() for i, v in enumerate(OPT.variables)})
    _atomic_savez(os.path.join(CKPT_DIR, 'last.npz'), arrs)
    if best_weights is not None:
        _atomic_savez(os.path.join(CKPT_DIR, 'best.npz'), {f'b{i}': w for i, w in enumerate(best_weights)})
    state['lr_ctl'] = lr_ctl.state()
    state['fingerprint'] = FINGERPRINT
    state['saved_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    _atomic_json(os.path.join(CKPT_DIR, 'state.json'), state)   # written last = commit point

def _read_state(d):
    sj = os.path.join(d, 'state.json')
    if not (os.path.isfile(sj) and os.path.isfile(os.path.join(d, 'last.npz'))):
        return None
    try:
        with open(sj) as f:
            return json.load(f)
    except Exception as e:
        print('  unreadable checkpoint', d, e)
        return None

def find_resume():
    if not RESUME:
        return None, None
    if RESUME_FROM:
        st = _read_state(RESUME_FROM)
        if st is None:
            raise FileNotFoundError(f'RESUME_FROM={RESUME_FROM!r} has no state.json + last.npz')
        return RESUME_FROM, st
    cands = [CKPT_DIR] + [os.path.dirname(p) for p in
                          glob.glob(f'/kaggle/input/**/{RUN_NAME}_ckpt/state.json', recursive=True)]
    best = (None, None)
    for d in cands:
        st = _read_state(d)
        if st is None:
            continue
        key = (st.get('epoch_done', 0), st.get('lr_ctl', {}).get('step', 0))
        if best[1] is None or key > (best[1].get('epoch_done', 0), best[1].get('lr_ctl', {}).get('step', 0)):
            best = (d, st)
    return best

def load_ckpt(d, st):
    fp = st.get('fingerprint', {})
    for k in ('preset', 'backbone', 'n_params', 'n_tvars'):
        if fp.get(k) != FINGERPRINT[k]:
            raise ValueError(f'checkpoint {d} was made with {k}={fp.get(k)!r} but this run has '
                             f'{k}={FINGERPRINT[k]!r}. Use the same PRESET/BACKBONE, or set RESUME=False.')
    if os.path.abspath(d) != os.path.abspath(CKPT_DIR):          # e.g. read-only /kaggle/input copy
        shutil.copytree(d, CKPT_DIR, dirs_exist_ok=True)
    z = np.load(os.path.join(CKPT_DIR, 'last.npz'))
    MODEL.set_weights([z[f'm{i}'] for i in range(len(MODEL.get_weights()))])
    if EMA:
        if 'e0' in z.files:
            for i, e in enumerate(EMA):
                e.assign(z[f'e{i}'])
        else:
            for e, v in zip(EMA, TVARS):
                e.assign(v)
    ov = OPT.variables
    if all(f'o{i}' in z.files and z[f'o{i}'].shape == tuple(ov[i].shape) for i in range(len(ov))):
        for i, v in enumerate(ov):
            v.assign(z[f'o{i}'])
    else:
        print('  optimizer state incompatible -> fresh optimizer moments (weights still restored)')
    if 'lr_ctl' in st:
        lr_ctl.load(st['lr_ctl'])
    print(f'  resumed from {d}: epoch_done={st["epoch_done"]}, best VAL={st.get("best_score")} '
          f'(epoch {st.get("best_epoch")}), optimizer step {int(OPT.iterations.numpy())}, lr {lr_ctl.lr:.2e}')
    return st

## 8. Train (resumes automatically, stops before the session limit)

In [ ]:
lr_ctl = LRController()
state = dict(epoch_done=0, best_score=None, best_epoch=0, best_src='', wait=0, stage=-1,
             finished=False, finish_kind='', history=[], sessions=0, last_epoch_sec=0.0, train_hours=0.0)
src_dir, src_state = find_resume()
if src_dir is not None:
    state.update(load_ckpt(src_dir, src_state))
else:
    print('  no checkpoint found -> training from scratch')
state['sessions'] += 1

# a finished run re-opens if the stop condition no longer holds with this session's config
# (e.g. MAX_EPOCHS or ES_PATIENCE was raised to train longer)
if state['finished']:
    still = ((state['finish_kind'] == 'early_stop' and state['wait'] >= ES_PATIENCE) or
             (state['finish_kind'] == 'max_epochs' and state['epoch_done'] >= MAX_EPOCHS))
    if not still:
        state['finished'] = False
        print(f'  run was finished ({state["finish_kind"]}) but MAX_EPOCHS={MAX_EPOCHS} / '
              f'ES_PATIENCE={ES_PATIENCE} allow more -> continuing')

DEADLINE = NB_T0 + SESSION_LIMIT_H * 3600 - FINAL_RESERVE_MIN * 60
print(f'session {state["sessions"]}: epochs done {state["epoch_done"]}/{MAX_EPOCHS}, '
      f'{(DEADLINE - time.time()) / 3600:.2f} h of training time left in this session')

stop_reason = 'already finished' if state['finished'] else None
ep = state['epoch_done'] + 1
while not state['finished'] and ep <= MAX_EPOCHS:
    est = state['last_epoch_sec'] * 1.15
    if time.time() + est > DEADLINE:
        stop_reason = f'session budget: next epoch needs ~{est/60:.0f} min'
        break
    si, K, tfr = stage_for_epoch(ep)
    counting = (si == len(CURRICULUM) - 1)            # early-stop patience counts in the last stage only
    if si != state['stage']:
        if state['stage'] >= 0:
            state['wait'], lr_ctl.wait = 0, 0
        state['stage'] = si
        print(f'== stage {si}: K={K} years per window, teacher-forcing={tfr:.2f}')

    B = global_batch_for(K)
    n_win = max(B, SAMPLES_PER_EPOCH // K)
    steps = n_win // B
    rng = np.random.default_rng([SEED, ep])
    tfr_t = tf.constant(tfr, tf.float32)
    t_ep, running, nb, partial = time.time(), 0.0, 0, False
    pbar = tqdm(make_dataset(K, n_win, B, rng), total=steps, desc=f'epoch {ep}/{MAX_EPOCHS} K={K} B={B}')
    for env, yy, af in pbar:
        lr_ctl.on_step()
        loss, _ = dist_step(env, yy, af, tfr_t)
        running += float(loss); nb += 1
        if nb % 50 == 0 or nb == steps:
            pbar.set_postfix(loss=f'{running/nb:.5f}', lr=f'{lr_ctl.lr:.1e}')
        if time.time() > DEADLINE:
            partial = True
            break
    pbar.close()
    if partial:                                      # keep the progress; this epoch is redone next session
        state['train_hours'] += (time.time() - t_ep) / 3600
        save_ckpt(state)
        stop_reason = f'session budget reached mid-epoch {ep} (it will be redone on resume)'
        break

    # ---- 40-year validation of the raw weights and of the EMA shadow ----
    raw_w = get_tvars()
    vscore, vh, va = scored_metric(rollout(x_val, val_y0, x_is_norm=True), val_true)
    tag, cand_w = 'raw', raw_w
    if EMA:
        ema_w = [e.numpy() for e in EMA]
        set_tvars(ema_w)
        es, eh, ea = scored_metric(rollout(x_val, val_y0, x_is_norm=True), val_true)
        set_tvars(raw_w)                             # training always continues from the raw weights
        if es < vscore:
            vscore, vh, va, tag, cand_w = es, eh, ea, 'ema', ema_w

    prev_best = np.inf if state['best_score'] is None else state['best_score']
    improved = vscore < prev_best - MIN_DELTA
    new_best = None
    if vscore < prev_best:
        state.update(best_score=float(vscore), best_epoch=ep, best_src=tag)
        new_best = cand_w
    lr_ctl.on_epoch(vscore, counting)
    if counting:
        state['wait'] = 0 if improved else state['wait'] + 1

    ep_sec = time.time() - t_ep
    state['last_epoch_sec'] = ep_sec
    state['train_hours'] += ep_sec / 3600
    state['epoch_done'] = ep
    state['history'].append(dict(epoch=ep, session=state['sessions'], K=K, tf_ratio=tfr, lr=lr_ctl.lr,
                                 train_loss=running / max(nb, 1), val_metric=float(vscore),
                                 val_mse_h=vh, val_mse_agb=va, src=tag, minutes=ep_sec / 60))
    if counting and state['wait'] >= ES_PATIENCE:
        state['finished'], state['finish_kind'] = True, 'early_stop'
        stop_reason = f'early stopping (best epoch {state["best_epoch"]})'
    elif ep >= MAX_EPOCHS:
        state['finished'], state['finish_kind'] = True, 'max_epochs'
        stop_reason = 'MAX_EPOCHS reached'
    save_ckpt(state, best_weights=new_best)
    print(f'  epoch {ep}: train {running/max(nb,1):.5f} | VAL-40y {vscore:.5f} ({tag}; h {vh:.4f}, agb {va:.4f}) '
          f'| lr {lr_ctl.lr:.1e} | {ep_sec/60:.1f} min' + ('  *best*' if new_best is not None else ''))
    ep += 1

print('\nstopped:', stop_reason or 'loop finished')
print(f'epochs done {state["epoch_done"]}/{MAX_EPOCHS} over {state["sessions"]} session(s), '
      f'{state["train_hours"]:.2f} h of training | best VAL {state["best_score"]} '
      f'(epoch {state["best_epoch"]}, {state["best_src"]})')
if not state['finished']:
    print(f'-> attach {RUN_NAME}_ckpt/ from this run\'s output and run again to continue at epoch '
          f'{state["epoch_done"] + 1}.')

best_path = os.path.join(CKPT_DIR, 'best.npz')
if os.path.isfile(best_path):
    zb = np.load(best_path)
    set_tvars([zb[f'b{i}'] for i in range(len(TVARS))])
    print('loaded best weights from', best_path)
else:
    print('no completed epoch yet -> using the current weights for the submission')
MODEL.save_weights(f'{OUT}/{RUN_NAME}.weights.h5')
hist_df = pd.DataFrame(state['history'])
hist_df.to_csv(f'{OUT}/{RUN_NAME}_history.csv', index=False)
hist_df.tail(10)

## 9. Validation with the best weights

In [ ]:
val_pred = rollout(x_val, val_y0, x_is_norm=True, desc='val rollout')
score, mse_h, mse_a = scored_metric(val_pred, val_true)
print(f'VAL metric = {score:.5f}  (height {mse_h:.5f}, agb {mse_a:.5f})')

## 10. Test rollout & submission (written every session)

In [ ]:
def load_test(scn):
    d = np.load(f'{COMP}/test_{scn}.npz')
    return d['test_x'].astype(np.float32), d['test_y0'].astype(np.float32)

sample_sub = pd.read_csv(f'{COMP}/sample_submission.csv')
parts, raw_preds = [], {}
for scn in SCENARIOS:
    tx, ty0 = load_test(scn)
    N = tx.shape[0]
    p = rollout(tx, np.transpose(ty0, (1, 0, 2)), desc=f'{scn} rollout')
    raw_preds[scn] = p
    ids = [f'{scn}_{site}_{age}' for age in SEED_AGES for site in range(N)]
    parts.append(pd.DataFrame({'id': ids,
                               'height': (p[:, :, 0] / Y_MEAN[0]).reshape(-1),
                               'agb':    (p[:, :, 1] / Y_MEAN[1]).reshape(-1)}))
    del tx, ty0; gc.collect()

sub = sample_sub[['id']].join(pd.concat(parts).set_index('id'), on='id')
assert len(sub) == len(sample_sub)
assert not sub[['height', 'agb']].isna().any().any(), 'NaNs / missing ids!'
sub.to_csv(f'{OUT}/submission.csv', index=False)
np.savez_compressed(f'{OUT}/{RUN_NAME}_preds.npz', val_pred=val_pred, **raw_preds)
print('wrote', f'{OUT}/submission.csv', 'and', f'{OUT}/{RUN_NAME}_preds.npz', '| rows', len(sub))
print(f'total session time {(time.time() - NB_T0) / 3600:.2f} h')

## 11. Notes

* `fusion_lite_preds.npz` has the same layout as the other runs (`val_pred`, `ssp126`, `ssp585`), so it can be
  added to `ensemble.py` / `carbonglobe-ensemble.ipynb`.
* Lighter still: lower `SAMPLES_PER_EPOCH`, or cap the curriculum at K=8. Heavier: `PRESET='full'` (works with
  resume, just needs more sessions).
* A second seed = same notebook with another `SEED` and `RUN_NAME`; blend the two `*_preds.npz`.